## Modelling of pion decay 

THe production of $\mu+$ from SM events is proceeded via four channels namely, CC_numu, CC_nuebar, NC_numu, NC_nuebar where CC and NC stands for charged and neutral current respectively. During hadronization,light mesons such as pions and kaons are also produced. These light mesons can further decay to $\mu^+$ + $\nu_{\mu}$. So modelling such events are important to study the effect it has on the background.

EXTRA NOTE: I have created an event file with only one event to study the code. Also, currently I have only included pion decay ( will add kaon once everything is confirmed)

In [1]:
import pandas as pd
import numpy as np
import math
import os
from skhep.math.vectors import LorentzVector, Vector3D

In [2]:
###Creation of input files
# Sample data for 3 events, each with pions and kaons
data = [
    # Event 1
    [1,0, 200.0, 211,  0.1, 0.2, 0.3, 50,213,90],
    [1,1, 200.0, 321,  -0.1, -0.2, -0.3, 40,213,90],
    [1,2, 200.0, 211,  0.2, 0.1, -0.1, 0.3,213,90],
    
    # Event 2
    [2,0, 300.0, 211,  0.3, 0.1, 0.2, 10,213,90],
    [2,1, 300.0, 321,  -0.2, -0.1, 0.1, 35,213,90],
    [2,2, 300.0, 211,  -0.3, 0.4, -0.2, 40,213,90],
    
    # Event 3
    [3,0, 400.0, 321,  0.5, 0.5, 0.4, 43,213,90],
    [3,1, 400.0, 211,  -0.3, -0.2, 0.1, 50,213,90],
    [3,2, 400.0, 321,  0.1, 0.2, -0.1, 51,213,90]
]
columns = ['ievent','iparticle', 'truth_energy', 'pid',  'px', 'py', 'pz', 'e', 'parent_pid1', 'parent_pid2']
df = pd.DataFrame(data, columns=columns)
df.to_csv("output_events_pion/events_1000.csv.zip", index=False)




In [3]:
data = pd.read_csv("output_events_pion/events_1000.csv.zip")
print(data)

   ievent  iparticle  truth_energy  pid   px   py   pz     e  parent_pid1  \
0       1          0         200.0  211  0.1  0.2  0.3  50.0          213   
1       1          1         200.0  321 -0.1 -0.2 -0.3  40.0          213   
2       1          2         200.0  211  0.2  0.1 -0.1   0.3          213   
3       2          0         300.0  211  0.3  0.1  0.2  10.0          213   
4       2          1         300.0  321 -0.2 -0.1  0.1  35.0          213   
5       2          2         300.0  211 -0.3  0.4 -0.2  40.0          213   
6       3          0         400.0  321  0.5  0.5  0.4  43.0          213   
7       3          1         400.0  211 -0.3 -0.2  0.1  50.0          213   
8       3          2         400.0  321  0.1  0.2 -0.1  51.0          213   

   parent_pid2  
0           90  
1           90  
2           90  
3           90  
4           90  
5           90  
6           90  
7           90  
8           90  


This function is taken from FORESEE to model the two body decay of $\pi^+$ / $K^+$ to $
\mu^+$ and $\nu_{\mu}$. It provides us with the produced particle's four momentum in lab frame


In [4]:
def twobody_decay(p0, m0, m1, m2):
    """
    Decays p0 -> p1 + p2 and returns the momenta of p1 and p2 in the lab frame.
    """
    phi = np.random.uniform(0, 2 * np.pi)
    costheta  = np.random.uniform(-1,1)
    
    zaxis = Vector3D(0, 0, 1)
    rotaxis = zaxis.cross(p0.vector).unit()
    rotangle = zaxis.angle(p0.vector)

    energy1 = (m0 * m0 + m1 * m1 - m2 * m2) / (2. * m0)
    energy2 = (m0 * m0 - m1 * m1 + m2 * m2) / (2. * m0)
    momentum1 = math.sqrt(energy1 * energy1 - m1 * m1)

    px1 = momentum1 * math.sqrt(1. - costheta * costheta) * np.cos(phi)
    py1 = momentum1 * math.sqrt(1. - costheta * costheta) * np.sin(phi)
    pz1 = momentum1 * costheta
    p1_rest = LorentzVector(-px1, -py1, -pz1, energy1)
    p2_rest = LorentzVector(px1, py1, pz1, energy2)

    if rotangle != 0:
        p1_rest = p1_rest.rotate(rotangle, rotaxis)
        p2_rest = p2_rest.rotate(rotangle, rotaxis)

    p1_lab = p1_rest.boost(-1. * p0.boostvector)
    p2_lab = p2_rest.boost(-1. * p0.boostvector)

    return p1_lab, p2_lab



To provide weight to the produced particle, we define the function calculate_decay_probability().The probability of decay of pion is given by 1 - $exp(-L/c\tau\times\gamma)$. Here, L is the distance to detector, c$\tau$ is the lifetime of pion and gamma is the boost factor caculated using the formula Energy_pion/mass_pion.

The decay_pion() function is defined to use two_body_decay() function specifically for pion decay. The mass specifications are:

pion_mass = 0.13957  # GeV/c^2


muon_mass = 0.10566  # GeV/c^2

neutrino_mass = 0.0  # GeV/c^2 (assumed negligible)

I have considered L as 10 m. Pion $\tau$ is 2.6033×10−8 seconds and hence we have $c\tau$ = 7.8 m



****##########NEWWWW ADDITION###### 
#added decay_prob in meson function. defined branching fraction and multiplied it to probability****

In [5]:


def decay_meson(px, py, pz, e, pid):
    """Simulate two-body decay of a pion (PID=211) or kaon (PID=321) into mu+ and neutrino."""
    
    # Define masses based on PID
    mass_dict = {
        211: 0.13957,  # Pion+
        321: 0.49367   # Kaon+
    }
    
    branching_fraction = {211: 0.99, 321: 0.63}
    c_tau_values = {211: 7.8, 321: 3.71}
    
    muon_mass = 0.10566  # GeV/c^2
    neutrino_mass = 0.0  # GeV/c^2 (assumed negligible)
    detector_distance =10    ###Distance to detector in meters
    
    
    c_tau = c_tau_values[pid]

    particle_mass = mass_dict[pid]
    
    

    # Particle 4-momentum
    p4_particle = LorentzVector(px, py, pz, e)
    
    #Decay probability calculation
    gamma = e / particle_mass
    
    mean_decay_length = c_tau * gamma
    
    decay_probability = (1 - np.exp(-detector_distance / mean_decay_length))*branching_fraction[pid]


    # Perform the decay
    muon, neutrino = twobody_decay(p4_particle, particle_mass, muon_mass, neutrino_mass)
    
    return muon, neutrino, decay_probability


Next, in the events dataframe I introduce first of all the decay products. I also introduce another column called weight which has entries 1 for all other particles and 1 - $exp(-L/c\tau\times\gamma)$ for the products. 

**IMPORTANT**: Here, in the code below I have tried to incorporate both pion and kaon decay. The code works the following way. If there is say one pion and one kaon event, it will make two copies of the event and decay one particle at once so in total there will be three events. One original and two copies with pion and kaon decay respectively.


**ADDED muon row to replce original pion/kaon row and added neutrino row with correct iparticle to the last place in the event so that when we are applying fix_dataframe everything is consistant**

In [6]:


def decay_and_flag(data, detector_distance):
    """Perform pion and kaon decay, generate event copies, and update event indices."""
    updated_data = []
    
    

    for ievent, original_event in enumerate(data['ievent'].unique(), start=1):
        event_data = data[data['ievent'] == original_event]  # Select all particles from this event
        
        # Select pions (PID=211) and kaons (PID=321) with E > 30 GeV
        decay_candidates = event_data[(event_data['pid'].isin([211, 321])) & (event_data['e'] > 30)].index
        
        # Store the original event first (ievent remains the same)
        event_copy = event_data.copy()
        event_copy['weight'] = 1.0  # All particles in the original event have weight 1
        event_copy['ievent'] = ievent  # Keep original event index
        updated_data.append(event_copy)

        # Loop over each decay candidate
        for i, pid_idx in enumerate(decay_candidates, start=1):
            event_copy = event_data.copy()
            event_copy['ievent'] = ievent + i  # Assign new ievent index
            
            max_iparticle = event_data['iparticle'].max()

            # Get particle row 
            particle_row = event_copy.loc[pid_idx]
            #event_copy = event_copy.drop(pid_idx)   ### (dont drop it instead relace it by decayed muon)

            # Get decay properties
            pid = particle_row['pid']
            

            # Set weight = 1 for all non-decay particles
            event_copy['weight'] = 1.0

            # Perform decay into muon and neutrino
            muon, neutrino,decay_probability = decay_meson(particle_row['px'], particle_row['py'], particle_row['pz'], particle_row['e'],pid)
            
            # Create muon row
            muon_row = particle_row.copy()
            muon_row['pid'] = -13
            muon_row['iparticle'] = particle_row['iparticle']   ##gets the same particle number as parent meson
            muon_row['px'], muon_row['py'], muon_row['pz'], muon_row['e'] = \
                muon.px, muon.py, muon.pz, muon.e
            muon_row['weight'] = decay_probability  # Set decay probability
            muon_row['parent_pid1'] = pid  # Parent ID = pion or kaon PID
            muon_row['parent_pid2'] = 90   # Second parent ID = 90
            
            
            # Replace the pion row with the muon row
            event_copy.loc[pid_idx] = muon_row

            
            
            # Create neutrino row
            neutrino_row = particle_row.copy()
            neutrino_row['pid'] = 14
            neutrino_row['iparticle'] = max_iparticle + 1  # Neutrino gets the next available iparticle number
            neutrino_row['px'], neutrino_row['py'], neutrino_row['pz'], neutrino_row['e'] = \
                neutrino.px, neutrino.py, neutrino.pz, neutrino.e
            neutrino_row['weight'] = decay_probability  # Set decay probability
            neutrino_row['parent_pid1'] = pid  # Parent ID = pion or kaon PID
            neutrino_row['parent_pid2'] = 90   # Second parent ID = 90

            
            # Add the neutrino at the end of the event copy
            event_copy = pd.concat([event_copy, pd.DataFrame([neutrino_row])], ignore_index=True)

            # Append the updated event to the final list
            updated_data.append(event_copy)
           

    return pd.concat(updated_data, ignore_index=True)





Here I comapre the input events file to the output after pion_decay. My input pions doesn't have energy>30GeV so we do not see any decay events

In [7]:
data = pd.read_csv("output_events_pion/events_1000.csv.zip")
print(data)
decay_and_flag(data, detector_distance=10)

   ievent  iparticle  truth_energy  pid   px   py   pz     e  parent_pid1  \
0       1          0         200.0  211  0.1  0.2  0.3  50.0          213   
1       1          1         200.0  321 -0.1 -0.2 -0.3  40.0          213   
2       1          2         200.0  211  0.2  0.1 -0.1   0.3          213   
3       2          0         300.0  211  0.3  0.1  0.2  10.0          213   
4       2          1         300.0  321 -0.2 -0.1  0.1  35.0          213   
5       2          2         300.0  211 -0.3  0.4 -0.2  40.0          213   
6       3          0         400.0  321  0.5  0.5  0.4  43.0          213   
7       3          1         400.0  211 -0.3 -0.2  0.1  50.0          213   
8       3          2         400.0  321  0.1  0.2 -0.1  51.0          213   

   parent_pid2  
0           90  
1           90  
2           90  
3           90  
4           90  
5           90  
6           90  
7           90  
8           90  


,ievent,iparticle,truth_energy,pid,px,py,pz,e,parent_pid1,parent_pid2,weight
0,1.0,0.0,200.0,211.0,0.100000,0.200000,0.300000,50.000000,213.0,90.0,1.000000
1,1.0,1.0,200.0,321.0,-0.100000,-0.200000,-0.300000,40.000000,213.0,90.0,1.000000
2,1.0,2.0,200.0,211.0,0.200000,0.100000,-0.100000,0.300000,213.0,90.0,1.000000
3,2.0,0.0,200.0,-13.0,-0.028835,-0.006020,0.001930,0.109706,211.0,90.0,0.003537
4,2.0,1.0,200.0,321.0,-0.100000,-0.200000,-0.300000,40.000000,213.0,90.0,1.000000
5,2.0,2.0,200.0,211.0,0.200000,0.100000,-0.100000,0.300000,213.0,90.0,1.000000
6,2.0,3.0,200.0,14.0,0.029114,0.006578,-0.001093,0.029868,211.0,90.0,0.003537
7,3.0,0.0,200.0,211.0,0.100000,0.200000,0.300000,50.000000,213.0,90.0,1.000000
8,3.0,1.0,200.0,-13.0,-0.043331,-0.182428,0.142426,0.258083,321.0,90.0,0.020613
9,3.0,2.0,200.0,211.0,0.200000,0.100000,-0.100000,0.300000,213.0,90.0,1.000000


**In the output there are now 10 events (original+copy). But the ievent is messed up coz I didnt use the fix_dataframe**
Next, I need to calculate the observables.Now i have another column named weight with the weight of the decay products which I just extract from the modified input events file

In [8]:
def fix_dataframe(data):

    # Initialize the column 'ievent' with zeros
    data['ievent'] = 0
    # Variable to keep track of the increment for column 'ievent'
    counter = 0
    # Loop through the DataFrame rows
    for index, row in data.iterrows():
        if row['iparticle'] == 0: counter += 1
        data.at[index, 'ievent'] = counter
    #return
    
    return data

def calculate_observables_with_flags(data):
    """Calculate observables, incorporating the decay flag and weights."""
    events = []
    grouped_data = data.groupby('ievent')
    
    

    for ievent, evt in grouped_data:

        
        e_mu_minus, e_mu_plus, has_charm, e_em, e_visible, ptx, pty, ht = 0, 0, 0, 0, 0, 0, 0, 0
        pt_mu_minus, pt_mu_plus, phi_mu_minus, phi_mu_plus = 0, 0, 0, 0
        event_weight = 1.0

        for _, row in evt.iterrows():
            #############CHANGE###############(added row['weight'])##################
            truth_energy, pid, parent_pid1,  weight = row.iloc[2], row.iloc[3], row.iloc[8],  row['weight']
            px, py, pz, e = row.iloc[4], row.iloc[5], row.iloc[6], row.iloc[7]

            ########CHANGE#################
            event_weight = min(event_weight,weight)
            
            #########CHANGE###############
            
            if abs(parent_pid1) in {411, 421, 431, 4122, 15, 521, 511, 531, 443, 5122, 4232, 4112, 5232}: 
                has_charm = 1

            if pid == 13: 
                if e > e_mu_minus:
                    e_mu_minus = e 
                    pt_mu_minus = np.sqrt(px**2 + py**2) 
                    phi_mu_minus = np.arctan2(py, px)
            elif pid == -13: 
                if e > e_mu_plus:
                    e_mu_plus = e 
                    pt_mu_plus = np.sqrt(px**2 + py**2) 
                    phi_mu_plus = np.arctan2(py, px)

            if abs(pid) in {22, 11}: 
                e_em += e 
            if abs(pid) not in {12, 14, 16, 39}: 
                e_visible += e 
                ptx += px 
                pty += py 
                ht += np.sqrt(px**2 + py**2) 

        pt_mis = np.sqrt(ptx**2 + pty**2)
        phi_vis = np.arctan2(pty, ptx)
        phi_mis = np.arctan2(-pty, -ptx)

        events.append([ievent, truth_energy, e_mu_minus, e_mu_plus, e_em, has_charm, e_visible, pt_mis, ht, pt_mu_minus, pt_mu_plus, phi_mu_minus, phi_mu_plus, phi_mis, phi_vis, event_weight])

    columns = ['ievent', 'truth_energy', 'e_mu_minus', 'e_mu_plus', 'e_em', 'has_charm', 'e_visible', 'pt_mis', 'ht', 'pt_mu_minus', 'pt_mu_plus', 'phi_mu_minus', 'phi_mu_plus', 'phi_mis', 'phi_vis', 'weight']
    observables = pd.DataFrame(events, columns=columns)

    return observables


Here, I calculate the observables after applying the decay_and_flag() function which creates copies of the main event with decays and its weighted accordingly as discussed above.

In [9]:
energies = [1000]
processes = ["pion"]

for process in processes: 
    for energy in energies:
        filename = f"output_events_{process}/events_{str(energy)}.csv.zip"

        if not os.path.isfile(filename):
            print(f"File not found: {filename}")
            continue

        print(process, energy)
        

        data = pd.read_csv(filename)
        
        if 0 not in data.index or np.isnan(data.loc[0, 'ievent']): 
            
            data = fix_dataframe(data)
        
############CHANGE###############################        
        data1 = decay_and_flag(data, detector_distance=10.0)
        data1 = fix_dataframe(data1)
        print(data1)
############CHANGE################################
        observables = calculate_observables_with_flags(data1)
        csv_file = f"output_events_{process}/new_observables_{str(energy)}.csv.zip"
        observables.to_csv(csv_file, index=False, compression='zip')



pion 1000
    ievent  iparticle  truth_energy    pid        px        py        pz  \
0        1        0.0         200.0  211.0  0.100000  0.200000  0.300000   
1        1        1.0         200.0  321.0 -0.100000 -0.200000 -0.300000   
2        1        2.0         200.0  211.0  0.200000  0.100000 -0.100000   
3        2        0.0         200.0  -13.0  0.008714 -0.015742  0.024185   
4        2        1.0         200.0  321.0 -0.100000 -0.200000 -0.300000   
5        2        2.0         200.0  211.0  0.200000  0.100000 -0.100000   
6        2        3.0         200.0   14.0 -0.008434  0.016301 -0.023348   
7        3        0.0         200.0  211.0  0.100000  0.200000  0.300000   
8        3        1.0         200.0  -13.0  0.196146 -0.117080 -0.059725   
9        3        2.0         200.0  211.0  0.200000  0.100000 -0.100000   
10       3        3.0         200.0   14.0 -0.197381  0.114611  0.056023   
11       4        0.0         300.0  211.0  0.300000  0.100000  0.200000   
12

In [10]:
data = pd.read_csv("output_events_pion/new_observables_1000.csv.zip")
data

,ievent,truth_energy,e_mu_minus,e_mu_plus,e_em,has_charm,e_visible,pt_mis,ht,pt_mu_minus,pt_mu_plus,phi_mu_minus,phi_mu_plus,phi_mis,phi_vis,weight
0,1,200.0,0,0.000000,0,0,90.300000,0.223607,0.670820,0,0.000000,0,0.000000,-2.677945,0.463648,1.000000
1,2,200.0,0,0.109876,0,0,40.409876,0.158792,0.465207,0,0.017993,0,-1.065257,2.324891,-0.816702,0.003537
2,3,200.0,0,0.258674,0,0,50.558674,0.528792,0.675645,0,0.228432,0,-0.538136,-2.788372,0.353221,0.020613
3,4,300.0,0,0.000000,0,0,85.000000,0.447214,1.039835,0,0.000000,0,0.000000,-1.107149,2.034444,1.000000
4,5,300.0,0,0.257579,0,0,50.257579,0.734108,1.051065,0,0.234837,0,1.666356,-1.540269,1.601324,0.023502
5,6,300.0,0,0.109932,0,0,45.109932,0.075368,0.565357,0,0.025522,0,2.912431,-3.064593,0.077000,0.004419
6,7,400.0,0,0.000000,0,0,144.000000,0.583095,1.291269,0,0.000000,0,0.000000,-2.111216,1.030377,1.000000
7,8,400.0,0,0.253904,0,0,101.253904,0.398644,0.807999,0,0.223838,0,-2.447397,0.367440,-2.774152,0.019197
8,9,400.0,0,0.109792,0,0,94.109792,0.921370,0.949369,0,0.018656,0,-0.750066,-2.299654,0.841938,0.003537
9,10,400.0,0,0.258269,0,0,93.258269,0.444131,1.300080,0,0.232418,0,2.474548,-1.609983,1.531609,0.016225
